# 25. Telemetry counters, windows, and alarms

**Level:** Advanced

**Learning outcome:** Keep units explicit, state rollover assumptions, bound sample storage, and prevent alert flapping.

**Prerequisites:** 03, 11-12, and 15: value models, protocols, and failure boundaries.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](24_Async_Objects_Concurrency_and_Thread_Safety.ipynb) | [Next notebook](26_Topology_Graphs_and_Address_Planning.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Model units, counter deltas, explicit rollover assumptions, windows, and priority.

**Always name units.** Bytes, bits, seconds, milliseconds, percentages, and fractions are not
interchangeable. For a monotonically increasing byte counter, a basic rate is
`(new_bytes - old_bytes) * 8 / elapsed_seconds`. Reject nonpositive elapsed time. A negative
delta is not a negative traffic rate; it may mean reset, rollover, an out-of-order sample,
or a different interface identity.

**Rollover requires assumptions.** A modulo calculation works only when the counter width is
known, there has been no reset, and the observation interval permits at most the number of
wraps your model can distinguish. Two samples alone generally cannot prove those assumptions.
The rollover exercise states a one-wrap, no-reset lab assumption explicitly.

**Sliding windows** retain recent samples using bounded storage such as deque(maxlen=...).
An average smooths variation but can hide spikes and depends on sample intervals. These
examples use equally weighted samples; irregular telemetry may need time weighting.

**Health decisions** need clear boundaries: does a load equal to the limit pass? Is missing
data unknown or unhealthy? A label such as OK should be backed by explicit conditions.
Hysteresis can avoid flapping when values oscillate near a threshold.

**Priority queues** can schedule more important checks first. Include a stable sequence
tie-breaker if task objects themselves are not orderable. Priority ordering is not fairness,
rate limiting, or bounded total work; a production scheduler must address those separately.

**Pause and explain:** Compute units for a rate and explain why two samples cannot distinguish every reset from rollover.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 086: Byte counters to bits per second

**Core concept:** A value object can make a unit conversion explicit and validated.

**Network-engineering purpose:** An interface rate should not silently treat bytes as bits or accept zero elapsed time.

### Read the implementation in this order
1. Validate positive elapsed seconds.
2. Reject a negative delta as reset or unordered input.
3. Convert byte delta to bits and divide by time.
4. Check a known arithmetic result.

**Common mistake / boundary:** This simple rate assumes both counters describe the same interface and no wrap occurred.

**Practice after the ten worked scenarios:** Return a structured unavailable result for resets rather than inventing a traffic rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0851 - Byte counters to bits per second: branch scenario

**Scenario 01/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `1000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [1]:
# Example 0851: Byte counters to bits per second - branch
case = {'counter': 1000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0852 - Byte counters to bits per second: campus scenario

**Scenario 02/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `2000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [2]:
# Example 0852: Byte counters to bits per second - campus
case = {'counter': 2000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0853 - Byte counters to bits per second: dc scenario

**Scenario 03/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `3000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [3]:
# Example 0853: Byte counters to bits per second - dc
case = {'counter': 3000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0854 - Byte counters to bits per second: wan scenario

**Scenario 04/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `4000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [4]:
# Example 0854: Byte counters to bits per second - wan
case = {'counter': 4000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0855 - Byte counters to bits per second: lab scenario

**Scenario 05/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `5000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [5]:
# Example 0855: Byte counters to bits per second - lab
case = {'counter': 5000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0856 - Byte counters to bits per second: wireless scenario

**Scenario 06/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `6000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [6]:
# Example 0856: Byte counters to bits per second - wireless
case = {'counter': 6000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0857 - Byte counters to bits per second: remote scenario

**Scenario 07/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `7000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [7]:
# Example 0857: Byte counters to bits per second - remote
case = {'counter': 7000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0858 - Byte counters to bits per second: core scenario

**Scenario 08/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `8000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [8]:
# Example 0858: Byte counters to bits per second - core
case = {'counter': 8000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0859 - Byte counters to bits per second: mgmt scenario

**Scenario 09/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `9000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [9]:
# Example 0859: Byte counters to bits per second - mgmt
case = {'counter': 9000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0860 - Byte counters to bits per second: backup scenario

**Scenario 10/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `10000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [10]:
# Example 0860: Byte counters to bits per second - backup
case = {'counter': 10000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


## Pattern 087: Counter rollover under an explicit assumption

**Core concept:** Modulo arithmetic can model one known-width wrap in a controlled lab.

**Network-engineering purpose:** A small synthetic counter makes rollover reasoning easy to inspect.

### Read the implementation in this order
1. Store the modulus from the bit width.
2. Validate both samples fit the width.
3. Compute the modular difference.
4. Test a deliberate wrap near the maximum.

**Common mistake / boundary:** A reset or multiple wraps cannot be identified from this result alone.

**Practice after the ten worked scenarios:** Add a maximum plausible delta and report ambiguity instead of claiming a reliable rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0861 - Counter rollover under an explicit assumption: branch scenario

**Scenario 01/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `0` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [11]:
# Example 0861: Counter rollover under an explicit assumption - branch
case = {'small_counter': 0}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 6


### Example 0862 - Counter rollover under an explicit assumption: campus scenario

**Scenario 02/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `1` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 7
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [12]:
# Example 0862: Counter rollover under an explicit assumption - campus
case = {'small_counter': 1}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 7


### Example 0863 - Counter rollover under an explicit assumption: dc scenario

**Scenario 03/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `2` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 8
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [13]:
# Example 0863: Counter rollover under an explicit assumption - dc
case = {'small_counter': 2}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 8


### Example 0864 - Counter rollover under an explicit assumption: wan scenario

**Scenario 04/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `3` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 9
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [14]:
# Example 0864: Counter rollover under an explicit assumption - wan
case = {'small_counter': 3}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 9


### Example 0865 - Counter rollover under an explicit assumption: lab scenario

**Scenario 05/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `4` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [15]:
# Example 0865: Counter rollover under an explicit assumption - lab
case = {'small_counter': 4}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 10


### Example 0866 - Counter rollover under an explicit assumption: wireless scenario

**Scenario 06/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `5` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 11
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [16]:
# Example 0866: Counter rollover under an explicit assumption - wireless
case = {'small_counter': 5}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 11


### Example 0867 - Counter rollover under an explicit assumption: remote scenario

**Scenario 07/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `6` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 12
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [17]:
# Example 0867: Counter rollover under an explicit assumption - remote
case = {'small_counter': 6}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 12


### Example 0868 - Counter rollover under an explicit assumption: core scenario

**Scenario 08/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `7` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 13
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [18]:
# Example 0868: Counter rollover under an explicit assumption - core
case = {'small_counter': 7}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 13


### Example 0869 - Counter rollover under an explicit assumption: mgmt scenario

**Scenario 09/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `8` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 14
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [19]:
# Example 0869: Counter rollover under an explicit assumption - mgmt
case = {'small_counter': 8}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 14


### Example 0870 - Counter rollover under an explicit assumption: backup scenario

**Scenario 10/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `9` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 15
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [20]:
# Example 0870: Counter rollover under an explicit assumption - backup
case = {'small_counter': 9}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 15


## Pattern 088: Bounded sliding sample window

**Core concept:** A deque limits retained history while the object exposes a useful query.

**Network-engineering purpose:** A telemetry collector can keep a fixed number of recent observations without unbounded growth.

### Read the implementation in this order
1. Initialize deque with a positive maxlen.
2. Append more samples than the window size.
3. Compute a mean only when data exists.
4. Check only the most recent samples remain.

**Common mistake / boundary:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

**Practice after the ten worked scenarios:** Expose minimum and maximum without returning the internal deque.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0871 - Bounded sliding sample window: branch scenario

**Scenario 01/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `20` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [21]:
# Example 0871: Bounded sliding sample window - branch
case = {'load': 20}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0872 - Bounded sliding sample window: campus scenario

**Scenario 02/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `90` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [22]:
# Example 0872: Bounded sliding sample window - campus
case = {'load': 90}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0873 - Bounded sliding sample window: dc scenario

**Scenario 03/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `80` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [23]:
# Example 0873: Bounded sliding sample window - dc
case = {'load': 80}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0874 - Bounded sliding sample window: wan scenario

**Scenario 04/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `0` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [24]:
# Example 0874: Bounded sliding sample window - wan
case = {'load': 0}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0875 - Bounded sliding sample window: lab scenario

**Scenario 05/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `100` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [25]:
# Example 0875: Bounded sliding sample window - lab
case = {'load': 100}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean:

 20.0


### Example 0876 - Bounded sliding sample window: wireless scenario

**Scenario 06/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `65` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [26]:
# Example 0876: Bounded sliding sample window - wireless
case = {'load': 65}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0877 - Bounded sliding sample window: remote scenario

**Scenario 07/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `71` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [27]:
# Example 0877: Bounded sliding sample window - remote
case = {'load': 71}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0878 - Bounded sliding sample window: core scenario

**Scenario 08/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `50` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [28]:
# Example 0878: Bounded sliding sample window - core
case = {'load': 50}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0879 - Bounded sliding sample window: mgmt scenario

**Scenario 09/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `95` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [29]:
# Example 0879: Bounded sliding sample window - mgmt
case = {'load': 95}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0880 - Bounded sliding sample window: backup scenario

**Scenario 10/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `10` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [30]:
# Example 0880: Bounded sliding sample window - backup
case = {'load': 10}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


## Pattern 089: Hysteresis avoids threshold flapping

**Core concept:** Different enter and clear thresholds make a stateful alarm deliberate.

**Network-engineering purpose:** A utilization alarm can remain active until load falls far enough below the alarm boundary.

### Read the implementation in this order
1. Validate low is below high.
2. Enter alarm only at the high threshold.
3. Retain the state between thresholds.
4. Clear at the low threshold and verify the transition sequence.

**Common mistake / boundary:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

**Practice after the ten worked scenarios:** Add an UNKNOWN state for stale observations rather than forcing a healthy result.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0881 - Hysteresis avoids threshold flapping: branch scenario

**Scenario 01/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [31]:
# Example 0881: Hysteresis avoids threshold flapping - branch
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0882 - Hysteresis avoids threshold flapping: campus scenario

**Scenario 02/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [32]:
# Example 0882: Hysteresis avoids threshold flapping - campus
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0883 - Hysteresis avoids threshold flapping: dc scenario

**Scenario 03/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [33]:
# Example 0883: Hysteresis avoids threshold flapping - dc
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0884 - Hysteresis avoids threshold flapping: wan scenario

**Scenario 04/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `60` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [34]:
# Example 0884: Hysteresis avoids threshold flapping - wan
case = {'threshold': 60}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0885 - Hysteresis avoids threshold flapping: lab scenario

**Scenario 05/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `90` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [35]:
# Example 0885: Hysteresis avoids threshold flapping - lab
case = {'threshold': 90}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0886 - Hysteresis avoids threshold flapping: wireless scenario

**Scenario 06/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `65` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [36]:
# Example 0886: Hysteresis avoids threshold flapping - wireless
case = {'threshold': 65}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0887 - Hysteresis avoids threshold flapping: remote scenario

**Scenario 07/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `70` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [37]:
# Example 0887: Hysteresis avoids threshold flapping - remote
case = {'threshold': 70}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0888 - Hysteresis avoids threshold flapping: core scenario

**Scenario 08/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `75` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [38]:
# Example 0888: Hysteresis avoids threshold flapping - core
case = {'threshold': 75}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]

### Example 0889 - Hysteresis avoids threshold flapping: mgmt scenario

**Scenario 09/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `85` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [39]:
# Example 0889: Hysteresis avoids threshold flapping - mgmt
case = {'threshold': 85}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]

### Example 0890 - Hysteresis avoids threshold flapping: backup scenario

**Scenario 10/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `50` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [40]:
# Example 0890: Hysteresis avoids threshold flapping - backup
case = {'threshold': 50}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]

## Pattern 090: Priority queue with a stable tie-breaker

**Core concept:** An explicit sequence number avoids comparing non-orderable task objects.

**Network-engineering purpose:** Equal-priority health checks should keep a predictable submission order.

### Read the implementation in this order
1. Store priority, sequence, and payload in each heap item.
2. Increment sequence per submission.
3. Pop through the queue API.
4. Check priority order and stability for a tie.

**Common mistake / boundary:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

**Practice after the ten worked scenarios:** Add an aging policy that eventually promotes waiting checks.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0891 - Priority queue with a stable tie-breaker: branch scenario

**Scenario 01/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [41]:
# Example 0891: Priority queue with a stable tie-breaker - branch
case = {'hostname': 'branch-edge-01'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['branch-edge-01', 'second-critical', 'routine']


### Example 0892 - Priority queue with a stable tie-breaker: campus scenario

**Scenario 02/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [42]:
# Example 0892: Priority queue with a stable tie-breaker - campus
case = {'hostname': 'campus-access-02'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['campus-access-02', 'second-critical', 'routine']


### Example 0893 - Priority queue with a stable tie-breaker: dc scenario

**Scenario 03/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [43]:
# Example 0893: Priority queue with a stable tie-breaker - dc
case = {'hostname': 'dc-leaf-03'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['dc-leaf-03', 'second-critical', 'routine']


### Example 0894 - Priority queue with a stable tie-breaker: wan scenario

**Scenario 04/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [44]:
# Example 0894: Priority queue with a stable tie-breaker - wan
case = {'hostname': 'wan-border-04'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['wan-border-04', 'second-critical', 'routine']


### Example 0895 - Priority queue with a stable tie-breaker: lab scenario

**Scenario 05/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [45]:
# Example 0895: Priority queue with a stable tie-breaker - lab
case = {'hostname': 'lab-router-05'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['lab-router-05', 'second-critical', 'routine']


### Example 0896 - Priority queue with a stable tie-breaker: wireless scenario

**Scenario 06/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [46]:
# Example 0896: Priority queue with a stable tie-breaker - wireless
case = {'hostname': 'wireless-dist-06'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['wireless-dist-06', 'second-critical', 'routine']


### Example 0897 - Priority queue with a stable tie-breaker: remote scenario

**Scenario 07/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [47]:
# Example 0897: Priority queue with a stable tie-breaker - remote
case = {'hostname': 'remote-gateway-07'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['remote-gateway-07', 'second-critical', 'routine']


### Example 0898 - Priority queue with a stable tie-breaker: core scenario

**Scenario 08/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [48]:
# Example 0898: Priority queue with a stable tie-breaker - core
case = {'hostname': 'core-router-08'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['core-router-08', 'second-critical', 'routine']


### Example 0899 - Priority queue with a stable tie-breaker: mgmt scenario

**Scenario 09/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [49]:
# Example 0899: Priority queue with a stable tie-breaker - mgmt
case = {'hostname': 'mgmt-switch-09'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['mgmt-switch-09', 'second-critical', 'routine']


### Example 0900 - Priority queue with a stable tie-breaker: backup scenario

**Scenario 10/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [50]:
# Example 0900: Priority queue with a stable tie-breaker - backup
case = {'hostname': 'backup-edge-10'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['backup-edge-10', 'second-critical', 'routine']


## Check your understanding: Telemetry counters, windows, and alarms

**You should now be able to:** Keep units explicit, state rollover assumptions, bound sample storage, and prevent alert flapping.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.